In [0]:
SELECT
    COUNT(*) AS eligible_isbns,
    COUNT(DISTINCT ISBN) AS distinct_isbns
FROM eligible_books;

eligible_isbns,distinct_isbns
14535,14535


In [0]:
SELECT
    ISBN,
    COUNT(*) AS occurrences
FROM books
WHERE ISBN IN (
    SELECT ISBN
    FROM eligible_books
)
GROUP BY ISBN
HAVING COUNT(*) > 1
ORDER BY occurrences DESC;

ISBN,occurrences


In [0]:
%python
import requests

test_isbns = [
    "0618002235",  # The Two Towers
    "0345339711",  # The Two Towers - another edition
    "0345339738",  # The Return of the King
    "0618002243"   # The Return of the King - another edition
]

for isbn in test_isbns:
    url = f"https://openlibrary.org/isbn/{isbn}.json"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()

        print("ISBN:", isbn)
        print("Edition ID:", data.get("key"))
        print("Title:", data.get("title"))
        print("Works:", data.get("works"))
        print("-------------------")
    else:
        print("ISBN:", isbn, "NOT FOUND", response.status_code)

ISBN: 0618002235
Edition ID: /books/OL7603170M
Title: The Two Towers
Works: [{'key': '/works/OL27479W'}]
-------------------
ISBN: 0345339711
Edition ID: /books/OL51688554M
Title: The Two Towers
Works: [{'key': '/works/OL27479W'}]
-------------------
ISBN: 0345339738
Edition ID: /books/OL51708836M
Title: The Return of the King
Works: [{'key': '/works/OL27455W'}]
-------------------
ISBN: 0618002243
Edition ID: /books/OL31973528M
Title: The Return of the King
Works: [{'key': '/works/OL27455W'}]
-------------------


In [0]:
%python
eligible_isbns_df = spark.sql("""
    SELECT DISTINCT ISBN
    FROM eligible_books
    WHERE ISBN IS NOT NULL
""")

eligible_isbns = [
    row["ISBN"]
    for row in eligible_isbns_df.collect()
]

print("Number of ISBNs:", len(eligible_isbns))
print(eligible_isbns[:10])

Number of ISBNs: 14535
['038550120X', '0060517794', '0671537458', '0679776818', '0684867621', '0451166892', '0380711524', '0553572369', '3499230933', '0345443683']


In [0]:
%python
import requests
import time

results = []

for isbn in eligible_isbns[:100]:
    url = f"https://openlibrary.org/isbn/{isbn}.json"

    try:
        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            data = response.json()
            works = data.get("works", [])

            work_id = works[0]["key"] if works else None

            results.append({
                "isbn": isbn,
                "work_id": work_id,
                "openlibrary_title": data.get("title"),
                "status": "found"
            })

        elif response.status_code == 404:
            results.append({
                "isbn": isbn,
                "work_id": None,
                "openlibrary_title": None,
                "status": "not_found"
            })

        else:
            results.append({
                "isbn": isbn,
                "work_id": None,
                "openlibrary_title": None,
                "status": f"http_{response.status_code}"
            })

    except Exception:
        results.append({
            "isbn": isbn,
            "work_id": None,
            "openlibrary_title": None,
            "status": "error"
        })

    time.sleep(0.1)

In [0]:
%python
import pandas as pd

test_mapping = pd.DataFrame(results)

display(test_mapping)

isbn,work_id,openlibrary_title,status
038550120X,/works/OL76991W,A Painted House,found
0060517794,/works/OL265289W,Little Altars Everywhere,found
0671537458,/works/OL3263743W,Waiting to exhale,found
0679776818,/works/OL80181W,Birdsong,found
0684867621,/works/OL81612W,The Girl Who Loved Tom Gordon,found
0451166892,/works/OL1914022W,The Pillars of the Earth,found
0380711524,/works/OL16476559W,See Jane run,found
0553572369,/works/OL29524W,Pay Dirt (Mrs. Murphy Mysteries),found
3499230933,/works/OL3501126W,Adressat unbekannt,found
0345443683,/works/OL77819W,Blackwood Farm,found


In [0]:
%python
display(
    test_mapping["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)

status,count
found,100


In [0]:
%python
print(
    "ISBNs with Work ID:",
    test_mapping["work_id"].notna().sum(),
    "/",
    len(test_mapping)
)

ISBNs with Work ID: 99 / 100


In [0]:
SELECT
    COUNT(*) AS isbn_count,
    COUNT(DISTINCT b.`Book-Title`) AS distinct_titles,
    COUNT(DISTINCT b.`Book-Author`) AS distinct_authors
FROM (
    SELECT ISBN
    FROM book_quality_scores
    ORDER BY weighted_rating DESC
    LIMIT 1000
) q
JOIN books b
    ON q.ISBN = b.ISBN;

isbn_count,distinct_titles,distinct_authors
953,890,585


saw that we have 953 isbn's even though i asked for 1000 so i check if the 47 remaining got lost

In [0]:
WITH top_1000 AS (
    SELECT ISBN, weighted_rating
    FROM book_quality_scores
    ORDER BY weighted_rating DESC
    LIMIT 1000
)
SELECT
    COUNT(*) AS missing_from_books
FROM top_1000 q
LEFT JOIN books b
    ON q.ISBN = b.ISBN
WHERE b.ISBN IS NULL;

missing_from_books
47


In [0]:
WITH top_1000 AS (
    SELECT ISBN, weighted_rating
    FROM book_quality_scores
    ORDER BY weighted_rating DESC
    LIMIT 1000
)
SELECT
    q.ISBN,
    q.weighted_rating
FROM top_1000 q
LEFT JOIN books b
    ON q.ISBN = b.ISBN
WHERE b.ISBN IS NULL
ORDER BY q.weighted_rating DESC;

ISBN,weighted_rating
8826703132,9.32123076923077
0789254484,9.029333333333334
1566404398,9.022
0908802862,8.941090909090908
0006513220,8.897
0261103253,8.779333333333334
0747549508,8.759272727272727
3150000017,8.753777777777778
1563137704,8.7352
0552137030,8.730206896551726


47 of the top 1,000 ISBNs based on rating information could not be matched to the books metadata and were therefore excluded from analyses requiring book or author information.

In [0]:
CREATE OR REPLACE VIEW author_quality_stats AS
SELECT
    b.`Book-Author` AS author,

    COUNT(DISTINCT q.ISBN) AS number_of_eligible_editions,

    SUM(q.rating_count) AS total_ratings,

    AVG(q.weighted_rating) AS avg_weighted_rating,

    MAX(q.weighted_rating) AS best_book_score

FROM book_quality_scores q
INNER JOIN books b
    ON q.ISBN = b.ISBN

WHERE b.`Book-Author` IS NOT NULL

GROUP BY b.`Book-Author`;

In [0]:
SELECT *
FROM author_quality_stats
ORDER BY total_ratings DESC
LIMIT 30;

author,number_of_eligible_editions,total_ratings,avg_weighted_rating,best_book_score
Stephen King,170,4189,7.880259976040058,8.975999999999999
Nora Roberts,119,2754,7.577568206693309,8.7176
John Grisham,35,2433,7.527875168913287,8.565538461538463
James Patterson,44,2330,7.623229384722833,8.514666666666667
J. K. Rowling,23,1659,8.62219271608839,9.476857142857144
Mary Higgins Clark,60,1519,7.554807976046324,8.239428571428572
Janet Evanovich,28,1453,7.96178959283744,8.618580645161291
Dean R. Koontz,55,1429,7.595058006322663,8.33408
Sue Grafton,28,1174,7.723955701424399,8.243254237288134
Anne Rice,40,1159,7.371330914237298,8.117600000000001


In [0]:
SELECT *
FROM author_quality_stats
ORDER BY avg_weighted_rating DESC
LIMIT 30;

author,number_of_eligible_editions,total_ratings,avg_weighted_rating,best_book_score
Pamela E. Apkarian-Russell,1,11,9.272,9.272
Michael Wallis,1,11,9.0845,9.0845
Dalton Trumbo,1,23,9.084,9.084
Bathroom Readers Institute,1,7,9.029333333333334,9.029333333333334
ALEX HALEY,1,21,9.013538461538463,9.013538461538463
Robert N. Munsch,1,25,9.011733333333334,9.011733333333334
Brothers Hildebrandt,1,9,8.953714285714286,8.953714285714286
DC Talk,1,12,8.903058823529411,8.903058823529411
C. S Lewis,1,8,8.87323076923077,8.87323076923077
Boston Women's Health Book Collective,1,7,8.862666666666668,8.862666666666668


In [0]:
CREATE OR REPLACE VIEW author_quality_stats AS
SELECT
    b.`Book-Author` AS author,
    COUNT(DISTINCT q.ISBN) AS number_of_eligible_editions,
    SUM(q.rating_count) AS total_ratings,
    AVG(q.weighted_rating) AS avg_weighted_rating,
    MAX(q.weighted_rating) AS best_book_score
FROM book_quality_scores q
INNER JOIN books b
    ON q.ISBN = b.ISBN
WHERE b.`Book-Author` IS NOT NULL
GROUP BY b.`Book-Author`;

In [0]:
SELECT
    author,
    number_of_eligible_editions,
    total_ratings,
    ROUND(avg_weighted_rating, 2) AS avg_weighted_rating,
    ROUND(best_book_score, 2) AS best_book_score
FROM author_quality_stats
ORDER BY total_ratings DESC
LIMIT 30;

author,number_of_eligible_editions,total_ratings,avg_weighted_rating,best_book_score
Stephen King,170,4189,7.88,8.98
Nora Roberts,119,2754,7.58,8.72
John Grisham,35,2433,7.53,8.57
James Patterson,44,2330,7.62,8.51
J. K. Rowling,23,1659,8.62,9.48
Mary Higgins Clark,60,1519,7.55,8.24
Janet Evanovich,28,1453,7.96,8.62
Dean R. Koontz,55,1429,7.6,8.33
Sue Grafton,28,1174,7.72,8.24
Anne Rice,40,1159,7.37,8.12


In [0]:
SELECT
    author,
    number_of_eligible_editions,
    total_ratings,
    ROUND(avg_weighted_rating, 2) AS avg_weighted_rating,
    ROUND(best_book_score, 2) AS best_book_score
FROM author_quality_stats
ORDER BY avg_weighted_rating DESC
LIMIT 30;

author,number_of_eligible_editions,total_ratings,avg_weighted_rating,best_book_score
Pamela E. Apkarian-Russell,1,11,9.27,9.27
Dalton Trumbo,1,23,9.08,9.08
Michael Wallis,1,11,9.08,9.08
Bathroom Readers Institute,1,7,9.03,9.03
Robert N. Munsch,1,25,9.01,9.01
ALEX HALEY,1,21,9.01,9.01
Brothers Hildebrandt,1,9,8.95,8.95
DC Talk,1,12,8.9,8.9
C. S Lewis,1,8,8.87,8.87
Boston Women's Health Book Collective,1,7,8.86,8.86


In [0]:
SELECT
    percentile_approx(weighted_rating, 0.25) AS p25,
    percentile_approx(weighted_rating, 0.50) AS median,
    percentile_approx(weighted_rating, 0.75) AS p75,
    percentile_approx(weighted_rating, 0.90) AS p90,
    percentile_approx(weighted_rating, 0.95) AS p95,
    percentile_approx(weighted_rating, 0.99) AS p99,
    MIN(weighted_rating) AS min_score,
    MAX(weighted_rating) AS max_score
FROM book_quality_scores;

p25,median,p75,p90,p95,p99,min_score,max_score
7.3568,7.690133333333333,8.027076923076924,8.304727272727273,8.486545454545453,8.788727272727272,4.418689419795221,9.476857142857144


a high-quality edition as one whose weighted rating is in the top 10%, so:
\[
\boxed{\text{weighted\_rating} \ge 8.3047}
\]

In [0]:
CREATE OR REPLACE VIEW author_quality_stats AS
SELECT
    b.`Book-Author` AS author,

    COUNT(DISTINCT q.ISBN) AS number_of_eligible_editions,

    COUNT(DISTINCT CASE
        WHEN q.weighted_rating >= 8.3047 THEN q.ISBN
    END) AS high_quality_editions,

    ROUND(
        100.0 * COUNT(DISTINCT CASE
            WHEN q.weighted_rating >= 8.3047 THEN q.ISBN
        END)
        / COUNT(DISTINCT q.ISBN),
        2
    ) AS high_quality_percentage,

    SUM(q.rating_count) AS total_ratings,

    AVG(q.weighted_rating) AS avg_weighted_rating,

    MAX(q.weighted_rating) AS best_book_score

FROM book_quality_scores q
INNER JOIN books b
    ON q.ISBN = b.ISBN

WHERE b.`Book-Author` IS NOT NULL

GROUP BY b.`Book-Author`;

In [0]:
SELECT
    author,
    number_of_eligible_editions,
    high_quality_editions,
    high_quality_percentage,
    total_ratings,
    ROUND(avg_weighted_rating, 2) AS avg_weighted_rating,
    ROUND(best_book_score, 2) AS best_book_score
FROM author_quality_stats
ORDER BY high_quality_editions DESC, total_ratings DESC
LIMIT 30;

author,number_of_eligible_editions,high_quality_editions,high_quality_percentage,total_ratings,avg_weighted_rating,best_book_score
Stephen King,170,32,18.82,4189,7.88,8.98
J. R. R. Tolkien,39,24,61.54,390,8.34,9.38
J. K. Rowling,23,18,78.26,1659,8.62,9.48
Douglas Adams,45,18,40.00,609,8.13,9.03
C. S. Lewis,30,18,60.00,420,8.36,9.06
Bill Watterson,16,16,100.00,299,8.84,9.25
Neil Gaiman,32,15,46.88,749,8.26,8.81
Jane Austen,39,12,30.77,454,8.02,8.79
Gary Larson,22,12,54.55,306,8.35,8.92
Dr. Seuss,15,12,80.00,180,8.67,9.23


build the Author Quality Score
Author Quality = Average Book Quality + Quality Consistency + Evidence

where:
- avg_weighted_rating → how good their eligible editions are on average;
- high_quality_percentage → how consistently they produce high-quality results;
- total_ratings → how much user evidence supports the assessment.

In [0]:
SELECT
    percentile_approx(number_of_eligible_editions, 0.50) AS editions_p50,
    percentile_approx(number_of_eligible_editions, 0.75) AS editions_p75,
    percentile_approx(number_of_eligible_editions, 0.90) AS editions_p90,
    percentile_approx(number_of_eligible_editions, 0.95) AS editions_p95,
    percentile_approx(number_of_eligible_editions, 0.99) AS editions_p99,

    percentile_approx(total_ratings, 0.50) AS ratings_p50,
    percentile_approx(total_ratings, 0.75) AS ratings_p75,
    percentile_approx(total_ratings, 0.90) AS ratings_p90,
    percentile_approx(total_ratings, 0.95) AS ratings_p95,
    percentile_approx(total_ratings, 0.99) AS ratings_p99,

    COUNT(*) AS total_authors
FROM author_quality_stats;

editions_p50,editions_p75,editions_p90,editions_p95,editions_p99,ratings_p50,ratings_p75,ratings_p90,ratings_p95,ratings_p99,total_authors
1,2,6,10,26,10,25,78,163,516,4813


I used the same Bayesian-style idea as for books:
\[
AdjustedRate =
\frac{n}{n+m}P +
\frac{m}{n+m}C
\]
where:
- \(P\) = author's proportion of high-quality editions
- \(n\) = number of eligible editions
- \(C\) = overall proportion of high-quality editions
- \(m\) = evidence threshold

We already defined high-quality as the top 10%, so \(C \approx 0.10\).

In [0]:
CREATE OR REPLACE VIEW author_adjusted_stats AS
SELECT
    *,
    
    (
        (number_of_eligible_editions / 
            (number_of_eligible_editions + 6.0))
        * (high_quality_percentage / 100.0)
        +
        (6.0 / 
            (number_of_eligible_editions + 6.0))
        * 0.10
    ) AS adjusted_high_quality_rate

FROM author_quality_stats;

In [0]:
SELECT
    author,
    number_of_eligible_editions,
    high_quality_editions,
    high_quality_percentage,
    ROUND(adjusted_high_quality_rate * 100, 2) 
        AS adjusted_high_quality_percentage,
    total_ratings,
    ROUND(avg_weighted_rating, 2) AS avg_weighted_rating
FROM author_adjusted_stats
ORDER BY adjusted_high_quality_rate DESC
LIMIT 30;

author,number_of_eligible_editions,high_quality_editions,high_quality_percentage,adjusted_high_quality_percentage,total_ratings,avg_weighted_rating
Bill Watterson,16,16,100.00,75.45,299,8.84
J. K. Rowling,23,18,78.26,64.14,1659,8.62
Dr. Seuss,15,12,80.00,60.00,180,8.67
Antoine de Saint-ExupÃ©ry,9,8,88.89,57.33,156,8.64
J. R. R. Tolkien,39,24,61.54,54.67,390,8.34
C. S. Lewis,30,18,60.00,51.67,420,8.36
E. B. White,7,6,85.71,50.77,145,8.54
L.M. MONTGOMERY,9,7,77.78,50.67,196,8.54
DIANA GABALDON,11,8,72.73,50.59,307,8.34
Lewis Carroll,8,6,75.00,47.14,68,8.35


I selected six editions as the shrinkage parameter because it corresponds approximately to the 90th percentile of author representation in this dataset. This reduces the influence of authors represented by only one or two editions.

We have duplicate author identities but written differently, for example: Agatha Christie and AGATHA CHRISTIE

In [0]:
SELECT
    LOWER(TRIM(`Book-Author`)) AS normalized_author,
    COUNT(DISTINCT `Book-Author`) AS name_variants,
    COLLECT_SET(`Book-Author`) AS original_names,
    COUNT(DISTINCT ISBN) AS isbn_count
FROM books
WHERE `Book-Author` IS NOT NULL
GROUP BY LOWER(TRIM(`Book-Author`))
HAVING COUNT(DISTINCT `Book-Author`) > 1
ORDER BY name_variants DESC, isbn_count DESC;

normalized_author,name_variants,original_names,isbn_count
jean de la fontaine,4,"List(Jean De La Fontaine, Jean de la Fontaine, Jean De LA Fontaine, Jean de La Fontaine)",7
agatha christie,3,"List(Agatha Christie, AGATHA CHRISTIE, Agatha christie)",635
danielle steel,3,"List(Danielle Steel, DANIELLE STEEL, danielle Steel)",214
louis l'amour,3,"List(LOUIS L'AMOUR, Louis L'Amour, Louis L'amour)",210
anne mccaffrey,3,"List(ANNE MCCAFFREY, Anne McCaffrey, Anne Mccaffrey)",189
robert louis stevenson,3,"List(Robert louis Stevenson, ROBERT LOUIS STEVENSON, Robert Louis Stevenson)",135
jude deveraux,3,"List(jude deveraux, Jude Deveraux, JUDE DEVERAUX)",122
john le carre,3,"List(JOHN LE CARRE, John le Carre, John Le Carre)",113
phyllis reynolds naylor,3,"List(PHYLLIS REYNOLDS NAYLOR, Phyllis Reynolds Naylor, phyllis reynolds naylor)",80
tim lahaye,3,"List(Tim Lahaye, Tim LaHaye, TIM LAHAYE)",73


In [0]:
CREATE OR REPLACE VIEW author_quality_stats_clean AS
SELECT
    LOWER(TRIM(b.`Book-Author`)) AS author_key,

    INITCAP(LOWER(TRIM(b.`Book-Author`))) AS author,

    COUNT(DISTINCT q.ISBN) AS number_of_eligible_editions,

    COUNT(DISTINCT CASE
        WHEN q.weighted_rating >= 8.3047 THEN q.ISBN
    END) AS high_quality_editions,

    ROUND(
        100.0 *
        COUNT(DISTINCT CASE
            WHEN q.weighted_rating >= 8.3047 THEN q.ISBN
        END)
        / COUNT(DISTINCT q.ISBN),
        2
    ) AS high_quality_percentage,

    SUM(q.rating_count) AS total_ratings,

    AVG(q.weighted_rating) AS avg_weighted_rating,

    MAX(q.weighted_rating) AS best_book_score

FROM book_quality_scores q
INNER JOIN books b
    ON q.ISBN = b.ISBN

WHERE b.`Book-Author` IS NOT NULL
  AND TRIM(b.`Book-Author`) <> ''
  AND LOWER(TRIM(b.`Book-Author`)) <> 'unknown'

GROUP BY LOWER(TRIM(b.`Book-Author`));

“I normalized capitalization and surrounding whitespace to consolidate obvious author duplicates. I avoided aggressive fuzzy matching because similar author names do not necessarily represent the same person. With more time, I would use an external author identifier, such as an Open Library author ID, to perform entity resolution more reliably.”

In [0]:
CREATE OR REPLACE VIEW author_adjusted_stats_clean AS
SELECT
    *,

    (
        (number_of_eligible_editions /
            (number_of_eligible_editions + 6.0))
        * (high_quality_percentage / 100.0)

        +

        (6.0 /
            (number_of_eligible_editions + 6.0))
        * 0.10

    ) AS adjusted_high_quality_rate

FROM author_quality_stats_clean;

In [0]:
SELECT
    author,
    number_of_eligible_editions,
    high_quality_editions,
    high_quality_percentage,
    ROUND(adjusted_high_quality_rate * 100, 2)
        AS adjusted_high_quality_percentage,
    total_ratings,
    ROUND(avg_weighted_rating, 2) AS avg_weighted_rating
FROM author_adjusted_stats_clean
ORDER BY adjusted_high_quality_rate DESC
LIMIT 30;

author,number_of_eligible_editions,high_quality_editions,high_quality_percentage,adjusted_high_quality_percentage,total_ratings,avg_weighted_rating
Bill Watterson,16,16,100.00,75.45,299,8.84
J. K. Rowling,23,18,78.26,64.14,1659,8.62
Dr. Seuss,15,12,80.00,60.00,180,8.67
L.m. Montgomery,14,11,78.57,58.00,236,8.5
Antoine De Saint-exupã©ry,9,8,88.89,57.33,156,8.64
J. R. R. Tolkien,39,24,61.54,54.67,390,8.34
C. S. Lewis,30,18,60.00,51.67,420,8.36
E. B. White,7,6,85.71,50.77,145,8.54
Lewis Carroll,8,6,75.00,47.14,68,8.35
Shel Silverstein,6,5,83.33,46.67,109,8.7


In [0]:
CREATE OR REPLACE VIEW author_score_components AS
SELECT
    *,

    PERCENT_RANK() OVER (
        ORDER BY avg_weighted_rating
    ) AS quality_score,

    PERCENT_RANK() OVER (
        ORDER BY adjusted_high_quality_rate
    ) AS consistency_score,

    PERCENT_RANK() OVER (
        ORDER BY total_ratings
    ) AS evidence_score

FROM author_adjusted_stats_clean

WHERE LOWER(TRIM(author)) NOT LIKE '%not applicable%';

In [0]:
CREATE OR REPLACE VIEW author_final_scores AS
SELECT
    *,

    (
        0.50 * quality_score
        + 0.30 * consistency_score
        + 0.20 * evidence_score
    ) * 100 AS author_quality_score

FROM author_score_components;

In [0]:
SELECT
    author,
    number_of_eligible_editions,
    high_quality_editions,
    ROUND(high_quality_percentage, 1) AS high_quality_percentage,
    total_ratings,
    ROUND(avg_weighted_rating, 2) AS avg_book_quality,
    ROUND(adjusted_high_quality_rate * 100, 1)
        AS adjusted_high_quality_percentage,
    ROUND(author_quality_score, 2) AS author_quality_score
FROM author_final_scores
ORDER BY author_quality_score DESC
LIMIT 30;

author,number_of_eligible_editions,high_quality_editions,high_quality_percentage,total_ratings,avg_book_quality,adjusted_high_quality_percentage,author_quality_score
Bill Watterson,16,16,100.0,299,8.84,75.5,99.27
Harper Lee,4,4,100.0,262,8.83,46.0,99.06
J. K. Rowling,23,18,78.3,1659,8.62,64.1,98.96
Dr. Seuss,15,12,80.0,180,8.67,60.0,98.4
Antoine De Saint-exupã©ry,9,8,88.9,156,8.64,57.3,98.06
J.r.r. Tolkien,11,6,54.6,588,8.52,38.8,98.03
Shel Silverstein,6,5,83.3,109,8.7,46.7,97.86
L.m. Montgomery,14,11,78.6,236,8.5,58.0,97.52
E. B. White,7,6,85.7,145,8.54,50.8,97.32
Sue Monk Kidd,3,3,100.0,403,8.43,40.0,97.05


J.r.r. Tolkien — 98.03
J. R. R. Tolkien — 96.51
And similarly:
J. K. Rowling — 98.96
Joanne K. Rowling — 95.44
and:
Dr. Seuss — 98.40
Dr Seuss — 94.88
These are being treated as separate authors.
I would not manually merge them in the whole dataset now. That can become a rabbit hole. But it's important that you acknowledge it because otherwise someone looking at the Top 30 could immediately ask why Tolkien appears twice.

“I created an Author Quality Score combining three dimensions: average weighted book quality, consistency in producing highly rated editions, and the amount of rating evidence available. The components were converted to percentile ranks so that metrics with different scales could be combined. I weighted them 50%, 30%, and 20%, respectively, prioritizing quality over popularity.”

In [0]:
SELECT
    author,
    ROUND(author_quality_score, 2) AS author_quality_score
FROM author_final_scores
WHERE LOWER(TRIM(author)) NOT LIKE '%not applicable%'
ORDER BY author_quality_score DESC
LIMIT 20;

author,author_quality_score
Bill Watterson,99.27
Harper Lee,99.06
J. K. Rowling,98.96
Dr. Seuss,98.4
Antoine De Saint-exupã©ry,98.06
J.r.r. Tolkien,98.03
Shel Silverstein,97.86
L.m. Montgomery,97.52
E. B. White,97.32
Sue Monk Kidd,97.05


In [0]:
SELECT
    author,
    ROUND(author_quality_score, 2) AS author_quality_score
FROM author_final_scores
WHERE LOWER(TRIM(author)) NOT LIKE '%not applicable%'
ORDER BY author_quality_score DESC;

author,author_quality_score
Bill Watterson,99.27
Harper Lee,99.06
J. K. Rowling,98.96
Dr. Seuss,98.4
Antoine De Saint-exupã©ry,98.06
J.r.r. Tolkien,98.03
Shel Silverstein,97.86
L.m. Montgomery,97.52
E. B. White,97.32
Sue Monk Kidd,97.05
